# Lesson 14 Lab — PyTorch Pruning API and the Complete Mask Lifecycle

**Puzzle:** What changes in a module before and after `prune.remove`, and what must a rollback loader know?

This notebook is designed for a CUDA GPU and retains the output of a complete RTX 5090 run.


## Why this matters

PyTorch pruning is a reparameterization lifecycle: apply a method, combine or update masks, train while preserving the mask, serialize the expected keys, optionally materialize with `remove`, and verify loading. Confusing a hook-based checkpoint with a materialized one breaks reproducibility.


## 0. Predict before running

1. Predict parameter and buffer names after the first mask.
2. Predict sparsity after two iterative 25% pruning calls.
3. Predict whether `remove` restores the deleted weights.

For every answer, name the observation that would prove it wrong.


## 1. Name the concrete objects

One CUDA linear layer is inspected at five points: dense, first mask, iterated mask, optimizer update, and removal. Named parameters, buffers, forward pre-hooks, sparsity, and output drift are captured.

- Parameter and buffer names encode the checkpoint lifecycle stage.
- Iterative masks compose rather than reset by default.
- Removal makes pruning permanent in a dense parameter.


## 2. Derive the mechanism

After pruning, `weight_orig` is a parameter and `weight_mask` a buffer; the visible `weight` is computed before forward. Iterative pruning combines masks through a pruning container. Gradients update `weight_orig`, so effective masked values remain zero at forward even when underlying values change. `remove` replaces this pair with a materialized `weight` parameter and deletes the hook; it does not undo pruning.

Keep value sparsity, physical shape, representation, and runtime evidence separate.


## 3. Verify the execution environment

Inspect the next cell before running it: it asserts CUDA, fixes the seed, defines transparent timing/numerical helpers, and prints the GPU/PyTorch/CUDA record needed to interpret every output.


In [1]:
LESSON_NO = 14
LESSON_TITLE = 'PyTorch Pruning API and the Complete Mask Lifecycle'

from pathlib import Path
import copy, gzip, hashlib, importlib.util, io, json, math, random, shutil, statistics, sys
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260808 + LESSON_NO
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name,
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered:
        return float("nan")
    position = (len(ordered) - 1) * q
    lo, hi = math.floor(position), math.ceil(position)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - position) + ordered[hi] * (position - lo)

def cuda_times(fn, warmup=6, repeats=24):
    with torch.inference_mode():
        for _ in range(warmup):
            fn()
        torch.cuda.synchronize()
        samples = []
        for _ in range(repeats):
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            fn()
            end.record()
            end.synchronize()
            samples.append(float(start.elapsed_time(end)))
    return samples

def timing_summary(samples):
    return {
        "median_ms": float(statistics.median(samples)),
        "p95_ms": float(percentile(samples, 0.95)),
        "p99_ms": float(percentile(samples, 0.99)),
        "samples_ms": [float(x) for x in samples],
    }

def count_params(module):
    return int(sum(p.numel() for p in module.parameters()))

def zero_fraction(tensor):
    return float((tensor == 0).float().mean().item())

def magnitude_mask(tensor, sparsity):
    flat = tensor.detach().abs().flatten()
    prune_count = int(round(flat.numel() * float(sparsity)))
    prune_count = min(max(prune_count, 0), flat.numel())
    mask = torch.ones_like(flat)
    if prune_count:
        idx = torch.topk(flat, prune_count, largest=False).indices
        mask[idx] = 0
    return mask.view_as(tensor)

def exact_2_4_mask(weight):
    assert weight.shape[-1] % 4 == 0
    groups = weight.detach().abs().reshape(*weight.shape[:-1], -1, 4)
    keep = torch.topk(groups, 2, dim=-1, largest=True).indices
    mask = torch.zeros_like(groups)
    mask.scatter_(-1, keep, 1)
    return mask.reshape_as(weight)

def compliance_2_4(weight):
    groups = weight.detach().reshape(*weight.shape[:-1], -1, 4)
    return float(((groups != 0).sum(dim=-1) == 2).float().mean().item())

def tensor_metrics(reference, candidate):
    ref = reference.float()
    cand = candidate.float()
    delta = cand - ref
    return {
        "rmse": float(torch.sqrt(torch.mean(delta.square())).item()),
        "mae": float(torch.mean(delta.abs()).item()),
        "max_error": float(delta.abs().max().item()),
        "cosine": float(F.cosine_similarity(ref.flatten(), cand.flatten(), dim=0).item()),
    }

def spearman(a, b):
    a = torch.as_tensor(a, dtype=torch.float64)
    b = torch.as_tensor(b, dtype=torch.float64)
    ra = torch.empty_like(a)
    rb = torch.empty_like(b)
    ra[torch.argsort(a)] = torch.arange(a.numel(), dtype=torch.float64)
    rb[torch.argsort(b)] = torch.arange(b.numel(), dtype=torch.float64)
    ra -= ra.mean(); rb -= rb.mean()
    return float((ra @ rb / (ra.norm() * rb.norm() + 1e-12)).item())


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.12.0",
  "cuda_runtime": "13.0",
  "python": "3.12.13",
  "seed": 20260822
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | dense module and single-mask state |
| Candidate | iteratively pruned, trained, and materialized module |
| Held constant | module, optimizer rule, input/target, pruning calls, seed, and inspection points |
| Measurements | effective sparsity, parameter names, buffer names, hook count, loss, and removal drift |
| Evidence | `pytorch-gpu` |

**Experiment:** Apply iterative PyTorch masks, take a training step, remove the reparameterization, and audit every state transition.


## 5. Read the experiment code

The notebook queries public module inspection APIs rather than inferring state from printed tensors. It records the effective weight before and after an optimizer step, then compares forward output immediately around `remove`. This produces a loader-oriented lifecycle trace.

Do not execute until the code implements the frozen table above.


In [2]:
import torch.nn.utils.prune as prune
layer=nn.Linear(256,128,device=DEVICE); x=torch.randn(32,256,device=DEVICE); target=torch.randn(32,128,device=DEVICE)
prune.l1_unstructured(layer,"weight",amount=0.25)
first_sparsity=zero_fraction(layer.weight); first_params=sorted(n for n,_ in layer.named_parameters()); first_buffers=sorted(n for n,_ in layer.named_buffers())
prune.l1_unstructured(layer,"weight",amount=0.25)
iterated_sparsity=zero_fraction(layer.weight); before_output=layer(x).detach(); hooks_before=len(layer._forward_pre_hooks)
opt=torch.optim.SGD(layer.parameters(),lr=0.01); opt.zero_grad(); loss=F.mse_loss(layer(x),target); loss.backward(); opt.step(); trained_output=layer(x).detach(); post_step_sparsity=zero_fraction(layer.weight)
params_before=sorted(n for n,_ in layer.named_parameters()); buffers_before=sorted(n for n,_ in layer.named_buffers())
pre_remove=layer(x).detach(); prune.remove(layer,"weight"); post_remove=layer(x).detach()
metrics={"first_sparsity":first_sparsity,"iterated_sparsity":iterated_sparsity,"post_step_sparsity":post_step_sparsity,"first_parameters":first_params,"first_buffers":first_buffers,"parameters_before_remove":",".join(params_before),"buffers_before_remove":",".join(buffers_before),"hooks_before_remove":hooks_before,"parameters_after_remove":sorted(n for n,_ in layer.named_parameters()),"buffers_after_remove":sorted(n for n,_ in layer.named_buffers()),"hooks_after_remove":len(layer._forward_pre_hooks),"training_loss":float(loss.item()),"training_output_change":tensor_metrics(before_output,trained_output)["rmse"],"remove_max_error":float((pre_remove-post_remove).abs().max().item())}
analysis=(f"The first call produced {first_sparsity:.1%} sparsity and the second composed to {iterated_sparsity:.1%}. "
          f"Before removal, parameters were `{metrics['parameters_before_remove']}`, buffers were `{metrics['buffers_before_remove']}`, "
          f"and {hooks_before} pre-hook was active. `remove` left drift {metrics['remove_max_error']:.3e} and restored a materialized `weight` parameter.")


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| First-mask sparsity | 25.00% |
| Iterated sparsity | 43.75% |
| Parameters before remove | bias,weight_orig |
| Buffers before remove | weight_mask |
| Hooks before remove | 1 |
| Remove max drift | 0.000000 |


## 7. Interpret rather than merely print

The first call produced 25.0% sparsity and the second composed to 43.8%. Before removal, parameters were `bias,weight_orig`, buffers were `weight_mask`, and 1 pre-hook was active. `remove` left drift 0.000e+00 and restored a materialized `weight` parameter.

The result is bounded to the shapes, seed, packages, and evidence label printed here.


## 8. Keep the evidence label honest

This run is labeled **`pytorch-gpu`**. The tensors and operators executed on CUDA through PyTorch. Native sparse-kernel identity is not inferred unless a trace or backend artifact names it.

The next cell writes the canonical JSON artifact and prints the same payload.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 14,
    "title": 'PyTorch Pruning API and the Complete Mask Lifecycle',
    "environment": ENV,
    "evidence_label": 'pytorch-gpu',
    "metrics": metrics,
    "analysis": analysis,
    "conclusion": 'PyTorch masks are auditable training state; `remove` materializes them but does not create a sparse runtime format.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 14,
  "title": "PyTorch Pruning API and the Complete Mask Lifecycle",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.12.0",
    "cuda_runtime": "13.0",
    "python": "3.12.13",
    "seed": 20260822
  },
  "evidence_label": "pytorch-gpu",
  "metrics": {
    "first_sparsity": 0.25,
    "iterated_sparsity": 0.4375,
    "post_step_sparsity": 0.4375,
    "first_parameters": [
      "bias",
      "weight_orig"
    ],
    "first_buffers": [
      "weight_mask"
    ],
    "parameters_before_remove": "bias,weight_orig",
    "buffers_before_remove": "weight_mask",
    "hooks_before_remove": 1,
    "parameters_after_remove": [
      "bias",
      "weight"
    ],
    "buffers_after_remove": [],
    "hooks_after_remove": 0,
    "training_loss": 1.3789167404174805,
    "training_output_change": 0.0009972263360396028,
    "remove_max_error": 0.0
  },
  "analysis": "The first call produced 25.0% sparsity and the second composed 

## 9. Make the bounded decision

> PyTorch masks are auditable training state; `remove` materializes them but does not create a sparse runtime format.

**Acceptance/rollback:** Accept a checkpoint only when its expected lifecycle stage, key schema, load procedure, mask policy, and rollback artifact are documented and tested.

**Failure analysis:** Loading a hook checkpoint into a plain module yields missing or unexpected keys. Optimizer state can point at reparameterized objects. Removing too early can allow zeros to regrow during later unconstrained training. These are state-management failures, not pruning-score failures.


## 10. Extend the evidence

Save and reload both lifecycle variants in fresh modules, test optimizer resume, and add a schema version to the structured artifact.

The full evidence boundary and references are in [`README.md`](README.md).
